# Build and contract-test a tool provider

This credential-free tutorial implements the smallest useful `ToolProvider`, verifies its metadata and output contract, and executes a program against it.

In [ ]:
from treelang import AST, ToolOutput, ToolProvider
from treelang.testing import ToolCallContract, ToolProviderContract

In [ ]:
class GreetingProvider(ToolProvider):
    def __init__(self):
        super().__init__()
        self.tools = {
            "greet": {
                "name": "greet",
                "description": "Return a greeting.",
                "properties": {"name": {"type": "string"}},
            }
        }

    async def list_tools(self):
        return list(self.tools.values())

    async def call_tool(self, name, arguments):
        if name != "greet":
            raise ValueError(f"Unknown tool: {name}")
        return ToolOutput(content=f"Hello, {arguments['name']}!")

Run the reusable provider contract before connecting the provider to Arborist or AST execution.

In [ ]:
tool = {
    "name": "greet",
    "description": "Return a greeting.",
    "properties": {"name": {"type": "string"}},
}
provider = GreetingProvider()
contract = ToolProviderContract(
    tools=(tool,),
    calls=(ToolCallContract("greet", {"name": "Ada"}, "Hello, Ada!"),),
)
await contract.verify(provider)

In [ ]:
program = AST.parse(
    {
        "type": "program",
        "body": [
            {
                "type": "function",
                "name": "greet",
                "params": [{"type": "value", "name": "name", "value": "Ada"}],
            }
        ],
        "mode": "single",
        "schema_version": "1.0",
    }
)
result = await AST.eval(program, provider)
assert result == "Hello, Ada!"
result